 # Talking Heroes #
 Three Dungeon'n'Dragons Characters created in week 1 excersise take a talk in front of a mysteriously looking cave.<br>
 <i>Excersise based on Week 2 content (incl. LiteLLM, Gradio)</i>


In [68]:
# base imports, settings and initialization of variables 

import json
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from litellm import completion
from openai import OpenAI

# load environment variables
load_dotenv(override=True)
# MODELS = ["openai/gpt-5.4", "anthropic/claude-sonnet-4-6", "gemini/gemini-2.5-flash"]
MODELS = ["openai/gpt-5-mini", "openai/gpt-5.4", "openai/gpt-5-nano"]

# assign API keys from env and check if key exists
error_msg = []
if not (openai_api_key := os.getenv('OPENAI_API_KEY')) or not openai_api_key.startswith("sk-proj-"):
    error_msg.append("No OPEN API key was found or is invalid (doesn't start sk-proj-)")
if not (anthropic_api_key := os.getenv('ANTHROPIC_API_KEY')) or not anthropic_api_key.startswith("sk-ant-"):
    error_msg.append("No ANTHROPIC API key was found or is invalid (doesn't start sk-ant-")
if not (google_api_key := os.getenv('GOOGLE_API_KEY')):
    error_msg.append("No GOOGLE API key was found")
if error_msg:
    raise("\n".join(error_msg))
print("API keys found and look good so far!")

# openai = OpenAI()


API keys found and look good so far!


In [69]:
# read characters from text file
with open(os.path.join(os.getcwd(), "..", "data", "dnd_characters.txt"), "r", encoding="utf-8") as datafile:
    content = datafile.read()

# split content into 3 characters with AI :)
user_prompt = """You are provided with a text containing 3 different game characters. 
Characters are separated by text Character N:, where N is an ordinary number. Your task is to extract each character, 
pick name and a summary of each character describing personality and communication style with maximum 333 characters.
Output format must be JSON and only JSON with structure: 
[
    {
        "name": "name of character N",
        "description": "summary of character N"
    }
] 
If there would be more than 3 characters then take the first three only. 
If there would be less than 3 characters then return JSON and only JSON with structure:
[
    {
        "error": "Error in content - not enough characters"
    }
]
The text containing 3 characters is: """ + content

response = completion(
    model="openai/gpt-5-mini", 
    messages=[{'role': 'user', 'content': user_prompt}], 
    reasoning_effort="minimal"
    )
character_list = json.loads(response.choices[0].message.content)

if err := character_list[0].get('error', False):
    raise(character_list[0]['error'])



In [70]:
[(item['name'], item['description']) for item in character_list]

[('Edron "Don" Gearwhistle',
  'Pragmatic, curious rock gnome artificer and tech-entrepreneur. Geeky explainer who over-shares details, collects prototypes, and prefers evidence to drama. Confident presenter and teacher with a snarky, helpful tone and a tendency to overfit neat models to complex reality.'),
 ('Queen Elsbeth Alexandra of Balmoral',
  'Dignified protector aasimar paladin and monarchly noble. Measured, mannered, duty-first leader who values ceremony, law, and stability. Calm, composed communicator who soothes councils, favors protocol over spectacle, and hides emotion behind reserved competence.'),
 ('Arnold von Thal',
  'Goliath paladin and former strongman turned civic leader. Direct, confident, showman-like and competitive with a commanding presence. Pragmatic, duty-driven, blunt communicator who enjoys applause but carries past scandals and physical limits that complicate public life.')]

In [ ]:
# 1. create system prompt (universal for all 3 agents) and user prompts
# 2. run 9 rounds of conversation and show tokens/cents spending with LiteLLM 

import gradio as gr
import litellm
import logging

from litellm.caching.caching import CacheMode

litellm._turn_on_debug()
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

system_prompt = """ 
    <role>Act as character from famous table-top role playing game Dungeon and Dragons.</role> 
    <context>You are standing in front of a mysteriosly looking cave, hearing some noise from inside, 
    but due to darkness you cannot see what is there.</context> 
    <task>Your task is to chat with your teammates according to role details you get in user prompt and decide what to do.</task>
    <rules>
        <rule>Dialogues are prefixed with character Name to make it clear who says what.</rule>
        <rule>Characters may react each other marking partner with naming
            <example>"Alex, I totally agree with you but there is some issue...".</example>
        </rule>
        <rule>Response must be shorter than 333 characters.</rule>
    </rules>
"""



user_prompts = [
 f"Your name is {character_list[0]['name']}, you act as character with following description: {character_list[0]['description']}. Your task is to persuade your teammates to find any other way into the cave, because you FEEL the main entrace you are standing in front of is a trap. You are a snearky speaker who loves jokes.",
 f"Your name is {character_list[1]['name']}, you act as character with following description: {character_list[1]['description']}. Your task is to persuade your teammates to leave the cave, go to the nearest village and find someone you could provide a help, because it may improve your karma better than to attack some creatures and get their treasures. You are wise but pragmatic speaker.",
 f"Your name is {character_list[2]['name']}, you act as character with following description: {character_list[2]['description']}. Goal you want the most would be to run directly inside the cave, attack anything inside and be again the hero with the most powerful muscles and shiny smile. You commonly don't care about someone elses feelings or talks, but you know you are team of three and must get to a democratic agreement."
]


def lets_talk(turns=9):
    chat_history = ""
    spending = {}
    for i in range(turns):
        speaker_coordinate = i%3 
        response = completion(
            model=MODELS[speaker_coordinate], 
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompts[speaker_coordinate] + chat_history}],
                stream=True,
                stream_options={"include_usage": True}
        )
        
        # chat_history += f"{response.choices[0].message.content}\n\n"


        for chunk in response:
            if chunk.get('usage'):
                if not spending.get(MODELS[speaker_coordinate]):
                    spending[MODELS[speaker_coordinate]] = { 
                        "hero": character_list[speaker_coordinate]['name'],
                        "total_tokens": chunk.usage['total_tokens']
                        #"total_cost": response._hidden_params["response_cost"]*100
                    }
                else:
                    spending[MODELS[speaker_coordinate]]["total_tokens"] += chunk.usage['total_tokens']
                    #spending[MODELS[speaker_coordinate]]["total_cost"] += response._hidden_params["response_cost"]*100
            
            
            chat_history += f"{chunk.choices[0].delta.content}" if chunk.choices[0].delta.content else "" 
            yield spending, chat_history
        chat_history += "\n\n"


import gradio as gr

# turns_input_mb = gr.Number(9, label="Count of turns:")
# talks_output_mk = gr.Markdown("Pre-cave's Heroic talks:")
# spendings_output_mk = gr.Markdown("Costs of Heroic talks:")

# gr.Interface(
#     fn=lets_talk, inputs=turns_input_mb, outputs=[spendings_output_mk, talks_output_mk]
# ).launch()

with gr.Blocks() as blck:
    with gr.Row():
        with gr.Column(scale=1):
            turns_input_mb = gr.Number(9, label="Count of turns:")
            spendings_output_mk = gr.Json(label="Costs of Heroic talks:")
        with gr.Column(scale=3):
            talks_output_mk = gr.Markdown("Pre-cave's Heroic talks:")
    with gr.Row():
        run_btn = gr.Button().click(fn=lets_talk, inputs=turns_input_mb, outputs=[spendings_output_mk, talks_output_mk])


blck.launch()







* Running on local URL:  http://127.0.0.1:7893
* To create a public link, set `share=True` in `launch()`.


In [40]:
import gradio as gr

turns_input_mb = gr.Number(9, label="Count of turns:")
talks_output_mk = gr.Markdown("Pre-cave's Heroic talks:")
spendings_output_mk = gr.Markdown("Costs of Heroic talks:")

gr.Interface(
    fn=lets_talk, inputs=turns_input_mb, outputs=[spendings_output_mk, talks_output_mk]
).launch()
        
#display(Markdown(lets_talk()))

#print("\nChat spendings:\n", json.dumps(spending, indent=4))


* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


In [61]:
from textwrap import indent
from litellm import completion 


def chat():
  usage = None
  completion = completion(
    model="gpt-4o",
    messages=[
      {"role": "system", "content": "You are a helpful assistant."},
      {"role": "user", "content": "Hello!"}
    ],
    stream=True,
    stream_options={"include_usage": True}
  )

  for chunk in completion:
    usage = chunk.get("usage", False)
    if chunk.choices[0].delta.content or usage:
      yield chunk.choices[0].delta.content, usage
    

for x, y in chat():
  print(x, '-', y)




UnboundLocalError: cannot access local variable 'completion' where it is not associated with a value